In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import pandas as pd
import glob
import numpy as np

from torch.utils.data import Dataset, DataLoader, TensorDataset

In [36]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers=1, bidirectional=True):
        super().__init__()

        self.data_in = nn.Linear(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers,
                            bidirectional=True, batch_first=True)
        self.bidirectional = bidirectional
        self.hidden_dim = hidden_dim

    def forward(self, x):
        emb = self.data_in(x)
        outputs, (h_n, c_n) = self.lstm(emb)
        return outputs, (h_n, c_n)

class AttentionModule(nn.Module):
    def __init__(self, encoding_dim, decoding_dim, attention_dim):
        super().__init__()

        self.enc = nn.Linear(encoding_dim, attention_dim, bias=False)
        self.dec = nn.Linear(decoding_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, dec_hidden, encoding_ouputs, mask=None):
        enc_proj = self.enc(encoding_ouputs) 
        dec_proj = self.dec(dec_hidden).unsqueeze(1)
        scores = self.v(torch.tanh(enc_proj + dec_proj)).squeeze(1)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9) # how the hell did even people come out w all this LSTM/Transformer stuff

        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), encoding_ouputs).squeeze(1)
        return context, weights 
    
class Decoder(nn.Module):
    def __init__(self, encoding_dim, decoding_dim, attention_dim, out_dim=1, emb_dim=32):
        super().__init__()

        self.attnMod = AttentionModule(encoding_dim, decoding_dim, attention_dim) 
        self.lstmCell = nn.LSTMCell(encoding_dim + emb_dim, decoding_dim)
        self.input_proj = nn.Linear(1, emb_dim)
        self.out = nn.Linear(decoding_dim + encoding_dim, out_dim)

    def forward(self, encoding_outputs, dec_h, dec_c, iters, first_input=None, mask=None):
        batch = encoding_outputs.size(0)
        device = encoding_outputs.device
        predictions = []
        input_t = first_input if first_input is not None else torch.zeros(batch, 1, device=device)

        for i in range(iters):
            emb_in = self.input_proj(input_t)
            context, weights = self.attnMod(dec_h, encoding_outputs, mask)
            lstm_in = torch.cat([emb_in, context], dim=1)
            dec_h, dec_c = self.lstmCell(lstm_in, (dec_h, dec_c))
            out = self.out(torch.cat([dec_h, context], dim=1))
            predictions.append(out)
            input_t = out.detach()

class Wrapper(nn.Module):
    def __init__(self, input_dim, enc_emb, enc_hid, dec_hid, attn_dim, H, bidir=True):
        super().__init__()

        self.encoder = Encoder(input_dim, enc_emb, enc_hid, bidirectional=bidir)
        enc_dim = enc_hid * (2 if bidir else 1)
        self.h_proj = nn.Linear(enc_dim * 1, dec_hid)
        self.c_proj = nn.Linear(enc_dim * 1, dec_hid)
        self.decoder = Decoder(enc_dim, dec_hid, attn_dim, out_dim=1)
        self.H = H

    def forward(self, x, first_input=None):
        enc_out, (h_n, c_n) = self.encoder(x)
        batch = x.size(0)
        h_flat = h_n.permute(1, 0, 2).contiguous().view(batch, -1)
        c_flat = c_n.permute(1, 0, 2).contiguous().view(batch, -1)
        dec_h = torch.tanh(self.h_proj(h_flat))
        dec_c = torch.tanh(self.c_proj(c_flat))
        preds = self.decoder(enc_out, dec_h, dec_c, self.H, first_input=first_input)
        return preds

In [37]:
class StockDataset(Dataset):
    def __init__(self, name, df, input_len=25, pred_len=5):
        self.input_len=input_len
        self.pred_len=pred_len
        self.name = name
        
        c_prices = torch.tensor(df["Close"].values, dtype=torch.float32)
        self.prices = (c_prices - c_prices.min()) / (c_prices.max() - c_prices.min())
        self.rsi = torch.tensor(df["Rsi"].values,  dtype=torch.float32) / 100
        self.k = torch.tensor(df["%K"].values,  dtype=torch.float32) / 100
        self.d = torch.tensor(df["%D"].values,  dtype=torch.float32) / 100

        self.features = torch.stack([self.prices, self.rsi, self.k, self.d], dim=1)

    def __len__(self):
        return len(self.features) - self.input_len - self.pred_len
    
    def __repr__(self):
        return self.name + f" of len {self.__len__()}"
    
    def __getitem__(self, idx):
        x = self.features[idx:idx+self.input_len]
        y = self.prices[idx+self.input_len:idx+self.input_len+self.pred_len]

        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


In [38]:
with open("global_vars.json", 'r') as file:
    vars = json.load(file)

split_ratio = 0.8
files = glob.glob(f"stocks/{vars["PERIOD"]}/{vars["INTERVAL"]}/*.csv")
train_seqs = []
val_seqs = []

for file in files:
    name = file.split("/")[-1].replace(".csv", "")
    df = pd.read_csv(file)
    split_idx = int(len(df) * split_ratio)

    train_df = df.iloc[:split_idx]
    validation_df = df.iloc[split_idx:]

    train_set = StockDataset(name, train_df)
    val_set = StockDataset(name, validation_df)

    train_seqs.extend([train_set[i] for i in range(len(train_set))])
    val_seqs.extend([val_set[i] for i in range(len(val_set))])

/var/folders/0_/8m1my8650593bn29fpf5hsvc0000gn/T/ipykernel_35530/3139140489.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)
/var/folders/0_/8m1my8650593bn29fpf5hsvc0000gn/T/ipykernel_35530/3139140489.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)
/var/folders/0_/8m1my8650593bn29fpf5hsvc0000gn/T/ipykernel_35530/3139140489.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTens

In [ ]:
x_train, y_train = zip(*train_seqs)
x_val, y_val = zip(*val_seqs)

x_train = torch.stack(x_train)
y_train = torch.stack(y_train)

x_val = torch.stack(x_val)
y_val = torch.stack(y_val)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=False)
val_loader   = DataLoader(TensorDataset(x_val, y_val), batch_size=32, shuffle=False)

tensor([0.0248, 0.0325, 0.0286, 0.0287, 0.0301])
tensor([0.0947, 0.0933, 0.0896, 0.0935, 0.0916])
tensor([0.0898, 0.0865, 0.0854, 0.0900, 0.0871])
tensor([0.0983, 0.0971, 0.0950, 0.0961, 0.0966])
tensor([0.1218, 0.1237, 0.1246, 0.1214, 0.1209])
tensor([0.1319, 0.1264, 0.1263, 0.1235, 0.1137])
tensor([0.1239, 0.1229, 0.1206, 0.1205, 0.1061])
tensor([0.1263, 0.1253, 0.1266, 0.1244, 0.1244])
tensor([0.1415, 0.1429, 0.1420, 0.1424, 0.1433])
tensor([0.2059, 0.1974, 0.2030, 0.1977, 0.1982])
tensor([0.1743, 0.1713, 0.1759, 0.1678, 0.1715])
tensor([0.1750, 0.1710, 0.1711, 0.1708, 0.1734])
tensor([0.2023, 0.2020, 0.2047, 0.2038, 0.2040])
tensor([0.1882, 0.1906, 0.1903, 0.1915, 0.1865])
tensor([0.2190, 0.2229, 0.2203, 0.2035, 0.1948])
tensor([0.2073, 0.2149, 0.2202, 0.2447, 0.2439])
tensor([0.1971, 0.1925, 0.1928, 0.1907, 0.1948])
tensor([0.1733, 0.1703, 0.1730, 0.1755, 0.1686])
tensor([0.1570, 0.1600, 0.1580, 0.1554, 0.1587])
tensor([0.1899, 0.1950, 0.1983, 0.2011, 0.1992])
tensor([0.1947, 0.19